In [1]:
import os

cwd = os.getcwd()  # Get the current working directory (cwd)
files = os.listdir(cwd)  # Get all the files in that directory
print("Files in %r: %s" % (cwd, files))



print(cwd)


Files in '/Users/benjedlovec/billiken': ['.Rhistory', 'billiken.Rproj', '.DS_Store', 'DraftAssistant', 'hitter_projections_ros_250424.csv', 'Simulate_Draft_2025.nb.html', 'test', 'hitter_projections_20240313.csv', 'projected_final_standings.csv', 'hitter_projections_ros.csv', 'proj_impact.csv', 'hitter_projections_20240301.csv', 'cleaned_rosters.csv', 'pitcher_projections_ros.csv', 'Billiken_Expansion_Draft_Rankings.nb.html', 'Pre_Freeze_Rankings_2025.Rmd', 'Simulate_Draft_2025.Rmd', 'Draft_Rankings_2025.Rmd', 'README.md', 'InSeason_Rankings_2025.nb.html', 'rosters_250531.csv', 'ImportBillikenSheets.Rmd', 'pitcher_projections_20240313.csv', 'pitcher_projections_ros_250424.csv', 'ImportBillikenSheets.nb.html', 'rosters_250731.csv', '.gitignore', 'hitter_projections_2025.csv', 'projections_2025.csv', '.RData', 'pitcher_projections_20240301.csv', 'rosters.csv', 'Billiken_Draft_Rankings.nb.html', 'clean_rosters.ipynb', 'Billiken_Expansion_Draft_Rankings.Rmd', 'InSeason_Rankings_2025.Rmd', 

In [2]:
import pandas as pd
import re

# === Configuration ===
input_file = '/Users/benjedlovec/billiken/rosters_250731.csv'              # Input CSV filename
output_file = '/Users/benjedlovec/billiken/cleaned_rosters.csv'     # Output CSV filename

mlb_teams = ["SD", "SF", "Ari", "Col", "Mil", "Atl", "ChC", "NYM", "Phi", "Pit", "Mia", "Wsh", "StL", "Cin", "LAD"]
status_suffixes = ["IL7", "IL10", "IL15", "IL60", "DTD", "SSPD"]

df = pd.read_csv(input_file, header=None, names=["Col1", "Col2", "Col3"])

cleaned_data = []
current_team = None
skipped = 0

for index, row in df.iterrows():
    col1 = str(row["Col1"]).strip()
    col2 = str(row["Col2"]).strip()
    col3 = str(row["Col3"]).strip()

    # Detect team name and score
    if re.match(r".+\(\d+(\.\d+)? pts\)", col1):
        current_team = re.sub(r"\(\d+(\.\d+)? pts\)", "", col1).strip()
        continue

    if col1 in ["SLOT", ""] or col2 in ["PLAYER", "Empty"] or "View TeamPropose Trade" in col1:
        continue

    slot = col1
    acq = col3
    player_raw = col2.replace("\n", "").strip()

    # Try to find the first MLB abbreviation match inside the string
    team_match = None
    for team in mlb_teams:
        if team in player_raw:
            split_point = player_raw.find(team)
            if split_point > 0:
                team_match = team
                break

    if not team_match:
        print(f"[Skipped: no team match] Row {index}: '{player_raw}'")
        skipped += 1
        continue

    team = team_match
    split_index = player_raw.find(team)
    name_part = player_raw[:split_index].strip()
    rest_part = player_raw[split_index + len(team):].strip()

    # Remove suffixes from name and position, even if attached
    for suffix in status_suffixes:
        name_part = re.sub(suffix + r"$", "", name_part)
        rest_part = re.sub(suffix + r"$", "", rest_part)
        name_part = re.sub(suffix, "", name_part)
        rest_part = re.sub(suffix, "", rest_part)

    name_clean = name_part.strip()
    pos_clean = rest_part.strip()

    cleaned_data.append({
        "BillikenTeam": current_team,
        "Slot": slot,
        "Player": name_clean,
        "Team": team,
        "Position": pos_clean,
        "Acq": acq
    })

cleaned_df = pd.DataFrame(cleaned_data)
cleaned_df.to_csv(output_file, index=False)

print(f"✅ Done. {len(cleaned_df)} rows written to {output_file}.")
print(f"❌ Skipped rows: {skipped}")


[Skipped: no team match] Row 63: 'Randal GrichukKCOF, DH'
[Skipped: no team match] Row 391: 'nan'
[Skipped: no team match] Row 464: 'Steven MatzBosSP, RP'
✅ Done. 304 rows written to /Users/benjedlovec/billiken/cleaned_rosters.csv.
❌ Skipped rows: 3
[Skipped: no team match] Row 63: 'Randal GrichukKCOF, DH'
[Skipped: no team match] Row 391: 'nan'
[Skipped: no team match] Row 464: 'Steven MatzBosSP, RP'
✅ Done. 304 rows written to /Users/benjedlovec/billiken/cleaned_rosters.csv.
❌ Skipped rows: 3
